# Paper 3 — Notebook 02: Clean & Split (frozen foundation)

**MeltpoolNet → source-aware generalization study.** This notebook produces the *frozen* data
artifacts every downstream notebook (baselines, multi-task, uncertainty, SHAP, maps) reads.
Run it once; never regenerate splits after you start looking at model results.

**What it does**
1. Clone MeltpoolNet (if needed) and load both CSVs.
2. Drop junk columns; clean classification labels to the 4-class set (drop `spatter formation`, n=8).
3. Define feature sets **F1–F5** per file (columns differ between the two CSVs).
4. Apply leakage guards (no target-derived ratios / outcomes in features).
5. Keep NaNs (native-NaN models like XGBoost) — **no imputation** baked into the frozen data.
6. Build split indices: **V0** random, **V1** grouped-by-`paper ID`, **V2** leave-one-material-out.
7. Build multi-task availability masks (depth / width / class).
8. Freeze everything to `./artifacts/` and run sanity/leakage assertions.

**Design decisions already locked (from the Stage-0 audit):**
- Source key = `paper ID` (0 missing; the `paper` URL column is mostly empty — do **not** use it).
- Regression: primary target **depth** (1443), secondary **width** (1148); **length** → appendix only (322).
- Regression CV = **10-fold** grouped (low source concentration: top-5 groups hold ≤29%).
- Classification: 4 classes; CV = **5-fold** grouped (only 16 source groups).
- Multi-task heads = depth + width + class, trained on the **regression file** (786 class labels there),
  with **masked per-head loss** over the union of labelled rows (uses all labels, discards none).


## 0 · Setup

In [ ]:
# Full stack for the whole project (SHAP + conformal installed now so every notebook shares one env).
# On Colab/Kaggle this cell just works. Safe to re-run; only installs what's actually missing.
import importlib.util, subprocess, sys

# pip name -> import name (differ for a few packages)
PKGS = {'pandas':'pandas','numpy':'numpy','scikit-learn':'sklearn','pyarrow':'pyarrow',
        'xgboost':'xgboost','shap':'shap','mapie':'mapie','matplotlib':'matplotlib'}

missing = [pip_name for pip_name, imp in PKGS.items()
           if importlib.util.find_spec(imp) is None]

if missing:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *missing]
    r = subprocess.run(cmd)
    if r.returncode != 0:
        # some managed environments require this flag; retry once
        subprocess.run(cmd + ['--break-system-packages'])

# report what's available (don't hard-fail if an optional extra is absent)
for pip_name, imp in PKGS.items():
    ok = importlib.util.find_spec(imp) is not None
    print(f'  {"ok " if ok else "MISSING"} {pip_name}')
print('environment ready')

In [ ]:
import os, re, json, hashlib, subprocess
import numpy as np, pandas as pd
from sklearn.model_selection import (KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold)

SEED = 42
np.random.seed(SEED)
ART = 'artifacts'
os.makedirs(ART, exist_ok=True)

# clone data if not present
if not os.path.exists('MeltpoolNet/Data/meltpoolnet_regression.csv'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/BaratiLab/MeltpoolNet.git'], check=True)

REG_PATH = 'MeltpoolNet/Data/meltpoolnet_regression.csv'
CLS_PATH = 'MeltpoolNet/Data/meltpoolnet_classification.csv'
print('seed', SEED, '| artifacts ->', os.path.abspath(ART))

## 1 · Load raw

In [ ]:
reg = pd.read_csv(REG_PATH)
cls = pd.read_csv(CLS_PATH)
print('regression   :', reg.shape)
print('classification:', cls.shape)

## 2 · Drop junk columns

In [ ]:
def drop_junk(df):
    junk = [c for c in df.columns if c.startswith('Unnamed:') or str(c).strip()=='']
    for extra in ['comment']:
        if extra in df.columns: junk.append(extra)
    return df.drop(columns=junk), junk

reg, jr = drop_junk(reg)
cls, jc = drop_junk(cls)
print('dropped from regression   :', jr)
print('dropped from classification:', jc)

## 3 · Clean classification labels

Keep the canonical 4 melt-pool modes. `spatter formation` (n=8) is dropped: too few to learn or
to survive grouped CV. Integer-encode in a fixed order so label ids are stable across all notebooks.

In [ ]:
CLASSES = ['desirable','keyhole','LOF','balling']   # fixed id order: 0,1,2,3
CLS_ID = {c:i for i,c in enumerate(CLASSES)}

print('classification labels BEFORE:', dict(cls['meltpool shape'].value_counts(dropna=False)))
cls = cls[cls['meltpool shape'].isin(CLASSES)].copy()
cls['y_class'] = cls['meltpool shape'].map(CLS_ID).astype(int)
print('classification labels AFTER :', dict(cls['meltpool shape'].value_counts()))

# The regression file ALSO carries meltpool shape -> used for the multi-task class head.
reg_class_mask = reg['meltpool shape'].isin(CLASSES)
print('\nregression-file class labels (multi-task head):',
      dict(reg.loc[reg_class_mask,'meltpool shape'].value_counts()))

## 4 · Feature sets F1–F5 (defined per file — the two CSVs expose different columns)

**Leakage guard:** target-derived ratios (`d/l`, `d/w`, `l/w`) and outcomes (`porosity`,
`relative density`, `spatter`, `meltpool shape`, `paper ID`, `paper`) are **never** features.
`F5` adds `Material` as a categorical purely as a *memorization diagnostic* — never used to make a
generalization claim.

- **Regression** has `powder flowrate`, `E (J/mm)`, `E (J/mm3)`.
- **Classification** lacks those but has dimensionless ratios `p/lb, p/l, p/b2, p/b, vb, vl` instead.

In [ ]:
def comp_cols(df):
    return [c for c in df.columns if re.search(r'\(wt\.?%\)', c)]

# --- regression feature ladder ---
reg_F1 = ['Power','Velocity','powder flowrate','layer thickness','beam D','Hatch spacing']
reg_F2 = reg_F1 + ['density','Cp','k','melting T','absorption coefficient','minimum absorptivity']
reg_F3 = reg_F2 + ['E (J/mm)','E (J/mm3)']
reg_F4 = reg_F3 + comp_cols(reg)
reg_F5 = reg_F3 + ['Material']            # diagnostic only

# --- classification feature ladder ---
cls_F1 = ['Power','Velocity','Hatch spacing','layer thickness','beam D']
cls_F2 = cls_F1 + ['density','Cp','k','melting T','absorption coefficient','minimal absorptivity']
cls_F3 = cls_F2 + ['p/lb','p/l','p/b2','p/b','vb','vl']
cls_F4 = cls_F3 + comp_cols(cls)
cls_F5 = cls_F3 + ['Material']            # diagnostic only

REG_FEATURES = {'F1':reg_F1,'F2':reg_F2,'F3':reg_F3,'F4':reg_F4,'F5':reg_F5}
CLS_FEATURES = {'F1':cls_F1,'F2':cls_F2,'F3':cls_F3,'F4':cls_F4,'F5':cls_F5}

# verify every listed column exists
for tag, fd, df in [('reg',REG_FEATURES,reg),('cls',CLS_FEATURES,cls)]:
    for name, cols in fd.items():
        miss = [c for c in cols if c not in df.columns]
        assert not miss, f'{tag} {name} missing {miss}'
        print(f'{tag} {name}: {len(cols)} cols  (missing: {miss})')

# leakage assertion
LEAKY = {'d/l','d/w','l/w','depth of meltpool','width of melt pool','length of melt pool',
         'spatter','porosity','relative density','meltpool shape','paper ID','paper'}
for fd in (REG_FEATURES, CLS_FEATURES):
    for cols in fd.values():
        assert not (set(cols) & LEAKY), f'LEAK: {set(cols)&LEAKY}'
print('\nleakage guard passed: no target-derived or outcome columns in any feature set')

## 5 · Coerce feature dtypes, keep NaN (no imputation)

In [ ]:
def coerce_numeric(df, cols):
    for c in cols:
        if c != 'Material':
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

reg = coerce_numeric(reg, sorted(set(sum(REG_FEATURES.values(), []))))
cls = coerce_numeric(cls, sorted(set(sum(CLS_FEATURES.values(), []))))

reg = reg.reset_index(drop=True)
cls = cls.reset_index(drop=True)

# missingness snapshot for the richest feature set (F4)
print('regression F4 missingness (top 8):')
print(reg[reg_F4].isna().mean().sort_values(ascending=False).head(8).round(2).to_string())
print('\nNaNs are RETAINED — downstream models must be native-NaN (XGBoost) or impute inside a')
print('fold-fitted pipeline, never globally.')

## 6 · Targets & multi-task availability masks (regression file)

Masks let the multi-task model use every label: each head back-propagates only on rows where its
own target exists (masked loss over the union), instead of the 821-row complete-case intersection.

In [ ]:
reg['has_depth'] = reg['depth of meltpool'].notna()
reg['has_width'] = reg['width of melt pool'].notna()
reg['has_class'] = reg['meltpool shape'].isin(CLASSES)
reg['y_class']   = reg['meltpool shape'].map(CLS_ID)   # NaN where no class label

print(f"has_depth = {reg['has_depth'].sum()}")
print(f"has_width = {reg['has_width'].sum()}")
print(f"has_class = {reg['has_class'].sum()}")
print(f"depth&width (complete-case for single-vs-multi comparison) = "
      f"{(reg['has_depth']&reg['has_width']).sum()}")

## 7 · Build split indices

- **V0** — random CV (the optimistic foil).
- **V1** — grouped by `paper ID` (the honest, primary protocol).
- **V2** — leave-one-material-out over alloys with ≥40 labelled rows.

Fold id `-1` means "row not part of that target's evaluation set". Regression folds are built on the
depth- and width-labelled subsets separately.

In [ ]:
K_REG, K_CLS = 10, 5

def reg_splits(mask_col, k=K_REG):
    m = reg[mask_col].values
    idx = np.where(m)[0]
    groups = reg.loc[idx, 'paper ID'].values
    v0 = np.full(len(reg), -1); v1 = np.full(len(reg), -1)
    for f,(_,te) in enumerate(KFold(k, shuffle=True, random_state=SEED).split(idx)):
        v0[idx[te]] = f
    for f,(_,te) in enumerate(GroupKFold(n_splits=k).split(idx, groups=groups)):
        v1[idx[te]] = f
    return v0, v1

reg['v0_depth'], reg['v1_depth'] = reg_splits('has_depth')
reg['v0_width'], reg['v1_width'] = reg_splits('has_width')

def lomo_materials(mask_col, min_rows=40):
    vc = reg.loc[reg[mask_col], 'Material'].value_counts()
    return vc[vc >= min_rows].index.tolist()

LOMO_DEPTH = lomo_materials('has_depth')
LOMO_WIDTH = lomo_materials('has_width')
print('depth fold sizes  V0:', [int((reg['v0_depth']==f).sum()) for f in range(K_REG)])
print('depth fold sizes  V1:', [int((reg['v1_depth']==f).sum()) for f in range(K_REG)])
print('LOMO depth materials:', LOMO_DEPTH)
print('LOMO width materials:', LOMO_WIDTH)

# classification splits (stratified; grouped variant respects paper ID)
cy, cg = cls['y_class'].values, cls['paper ID'].values
cls['v0'] = -1; cls['v1'] = -1
for f,(_,te) in enumerate(StratifiedKFold(K_CLS, shuffle=True, random_state=SEED).split(cls, cy)):
    cls.loc[te,'v0'] = f
for f,(_,te) in enumerate(StratifiedGroupKFold(K_CLS, shuffle=True, random_state=SEED).split(cls, cy, groups=cg)):
    cls.loc[te,'v1'] = f
LOMO_CLS = cls['Material'].value_counts()[lambda s: s>=40].index.tolist()
print('\nclassification LOMO materials:', LOMO_CLS)

## 8 · Sanity & leakage assertions (must all pass before freezing)

In [ ]:
# 8a. no source-group appears in both train and test of any grouped fold
def assert_group_disjoint(df, fold_col, group_col, k):
    for f in range(k):
        te = set(df.loc[df[fold_col]==f, group_col])
        tr = set(df.loc[(df[fold_col]!=f) & (df[fold_col]!=-1), group_col])
        assert te.isdisjoint(tr), f'{fold_col}: group leak in fold {f}'
assert_group_disjoint(reg[reg['has_depth']], 'v1_depth', 'paper ID', K_REG)
assert_group_disjoint(reg[reg['has_width']], 'v1_width', 'paper ID', K_REG)
assert_group_disjoint(cls, 'v1', 'paper ID', K_CLS)

# 8b. every labelled row is assigned to exactly one eval fold
assert (reg.loc[reg['has_depth'],'v1_depth']>=0).all()
assert (reg.loc[reg['has_width'],'v1_width']>=0).all()
assert (cls['v1']>=0).all()

# 8c. balling-per-fold warning (data-driven, not a failure) — see notebook 03 for pooled OOF metric
print('classification V1 balling count per fold (thin folds are expected & handled via pooled OOF):')
for f in range(K_CLS):
    print(f'  fold {f}: balling =', int(((cls["v1"]==f)&(cls["y_class"]==CLS_ID["balling"])).sum()))
print('\nALL SANITY CHECKS PASSED')

## 9 · Freeze artifacts

In [ ]:
# cleaned frames (parquet keeps dtypes + NaNs)
reg.to_parquet(f'{ART}/reg_clean.parquet', index=False)
cls.to_parquet(f'{ART}/cls_clean.parquet', index=False)

feature_sets = {'regression': REG_FEATURES, 'classification': CLS_FEATURES}
with open(f'{ART}/feature_sets.json','w') as f:
    json.dump(feature_sets, f, indent=2)

meta = {
    'seed': SEED,
    'source_key': 'paper ID',
    'classes': CLASSES,
    'regression': {
        'primary_target': 'depth of meltpool',
        'secondary_target': 'width of melt pool',
        'appendix_target': 'length of melt pool',
        'k_folds': K_REG,
        'n_depth': int(reg['has_depth'].sum()),
        'n_width': int(reg['has_width'].sum()),
        'n_class_labels': int(reg['has_class'].sum()),
        'n_depth_and_width': int((reg['has_depth']&reg['has_width']).sum()),
        'lomo_depth_materials': LOMO_DEPTH,
        'lomo_width_materials': LOMO_WIDTH,
    },
    'classification': {
        'k_folds': K_CLS,
        'class_counts': {c:int((cls['y_class']==i).sum()) for c,i in CLS_ID.items()},
        'lomo_materials': LOMO_CLS,
        'note': 'grouped folds strand balling in some folds; use pooled out-of-fold predictions '
                'for macro-F1, not per-fold averaging.',
    },
    'data_commit': subprocess.run(['git','-C','MeltpoolNet','rev-parse','HEAD'],
                                  capture_output=True, text=True).stdout.strip(),
}
with open(f'{ART}/meta.json','w') as f:
    json.dump(meta, f, indent=2)

print('written:')
for p in ['reg_clean.parquet','cls_clean.parquet','feature_sets.json','meta.json']:
    print('  ', f'{ART}/{p}')
print('\nmeta.json:')
print(json.dumps(meta, indent=2))

## 10 · How downstream notebooks consume this

```python
import pandas as pd, json
reg = pd.read_parquet('artifacts/reg_clean.parquet')
cls = pd.read_parquet('artifacts/cls_clean.parquet')
FS  = json.load(open('artifacts/feature_sets.json'))

# depth regression, honest protocol, feature set F2:
sub  = reg[reg['has_depth']]
X    = sub[FS['regression']['F2']]      # NaNs retained -> XGBoost handles natively
y    = sub['depth of meltpool']
fold = sub['v1_depth']                  # grouped-by-paper-ID folds
# loop folds, nested-tune inside training groups only, predict held-out fold, pool OOF preds.
```

**Next — Notebook 03 (baselines & the optimism gap):** for depth, width, and classification, run
Ridge/RF/**XGBoost**/SVM across **V0 vs V1 vs V2** with nested grouped tuning, and report the
V0→V1 drop with group-level bootstrap CIs. That drop is the paper's headline result.
